# CASMI26 explained for humans: the task and the top public pipeline

I started this notebook to learn. With the number of seemingly AI-generated submissions and notebooks
in this competition, it has become hard to move through the notebook list and actually understand
what is going on. FPNets? ICEBERG and GLACIER? RRF fusion, popularity priors, N1A? Personally, I don't
get it, and trying to solve a task you barely understand is a slippery slope.

So I am using the Feynman technique: *test how well you understand something by explaining it in
simple terms*. This notebook is that explanation.

The plan: explain the task, then build up the top public solution one idea at a time and show what
gives it its edge. I will keep this notebook up to date as the public leaderboard moves, so you
can follow along; the changelog is at the end.

*Last updated 2026-10-29, built from commit `325c080`.*

In [1]:
import base64
import glob
import io
import re
import subprocess
import sys
from pathlib import Path

try:
    import rdkit  # noqa: F401
except ImportError:
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    whl = [w for w in glob.glob("/kaggle/input/**/rdkit-*.whl", recursive=True) if tag in w]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", whl[0]], check=True)

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import HTML, Markdown, display


def find_train():
    if Path("/kaggle/input").exists():
        return sorted(glob.glob("/kaggle/input/**/train.parquet", recursive=True))[0]
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "data/raw/train.parquet").exists():
            return str(d / "data/raw/train.parquet")
    raise FileNotFoundError("train.parquet not found")


def show(fig):
    """Display a figure as a centred SVG, like the LaTeX figures."""
    buf = io.BytesIO()
    fig.savefig(buf, format="svg", bbox_inches="tight")
    plt.close(fig)
    svg = buf.getvalue()
    width = round(float(re.search(rb'width="([\d.]+)pt"', svg)[1]) * 96 / 72)
    b64 = base64.b64encode(svg).decode()
    display(HTML(f'<div align="center" style="text-align:center"><img src="data:image/svg+xml;base64,{b64}" '
                 f'style="width:{width}px;max-width:100%;background:white"></div>'))


def spectrum_axes(r, title, xmax=None):
    """Stick plot of one spectrum; the y axis starts exactly at 0."""
    mz, inten = np.array(r["ms2_mzs"]), np.array(r["ms2_normalized_intensities"])
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.vlines(mz, 0, inten, lw=1.8, color="#4a6fa5", capstyle="butt")
    ax.axvline(r["precursor_mz"], ls="--", c="grey", lw=0.8)
    ax.set(xlabel="m/z", ylabel="relative intensity", title=title, ylim=(0, 1.05),
           xlim=(min(40, mz.min() - 5), xmax or r["precursor_mz"] + 15))
    ax.spines[["top", "right"]].set_visible(False)
    return fig, ax, mz, inten


lf = pl.scan_parquet(find_train())

# Pick one [M+H]+ molecule with a few spectra and a few dozen same-formula isomers, deterministically.
meta = (
    lf.select("inchikey14", "molecular_formula", "adduct")
    .filter(pl.col("adduct") == "[M+H]+")
    .group_by("inchikey14", "molecular_formula")
    .len()
    .filter(pl.col("len").is_between(3, 6))
    .sort("inchikey14")
    .collect()
)
iso = meta.group_by("molecular_formula").agg(pl.col("inchikey14").n_unique().alias("isomers"))
pick = (
    meta.join(iso, on="molecular_formula")
    .filter(pl.col("isomers").is_between(15, 40))
    .sample(1, seed=0)
    .row(0, named=True)
)
QUERY_IK14, FORMULA = pick["inchikey14"], pick["molecular_formula"]
rows = lf.filter(pl.col("molecular_formula") == FORMULA).collect()
query = rows.filter(pl.col("inchikey14") == QUERY_IK14)
spec = query.row(0, named=True)

## 1. Words you need

### InChIKey14

The competition needs a way to decide whether a guess is the right molecule. It uses the **InChIKey**:
a fixed-length code computed from a structure, in three blocks, for example
`QNAYBMKLOCPYGJ-REOHCLBHSA-N`. The first block (14 letters) encodes which atoms are connected to which;
the second block encodes stereochemistry (see below). A guess counts as correct when the first block,
the **InChIKey14**, matches the answer's.

The comparison is built so that two kinds of variation count as the same answer: **stereoisomers**
(only the first block is compared) and **tautomers** (structures are normalised before the key is
computed). Both are explained below.

Each pair of molecules below is shown the same way: a 2D drawing on top, its InChIKey14 underneath,
then a short description. A bracket joins the two keys and gives the verdict: green ✓ when the keys
match (the competition counts both drawings as the same answer), red ✗ when they differ. Red letters
show where two keys differ; grey parts of a key are ignored by the competition.

### Formula and isomers

The **formula** of a molecule counts every type of atom in it. For example, C2H6O means two carbon (C),
six hydrogen (H) and one oxygen (O) atoms. But one formula can correspond to different molecules,
because the same atoms can be connected in different ways:

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

Molecules with the same formula but different connections are **isomers**. They get different keys
(13 of the 14 letters differ here), so they are different answers. The mass spectrometer measures a
molecule's mass precisely enough to give its formula (next section); telling the isomers of that
formula apart from their spectra is the whole difficulty of this competition.

### Stereochemistry

Your two hands are both hands, yet one is the mirror image of the other. Molecules can be like that too.

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

**Stereoisomers** have the same atoms connected the same way, but arranged differently in 3D. In a 2D
drawing, a solid wedge points towards you and a hashed wedge points away. Their InChIKey14 is identical;
only the second block of the full key differs. MS/MS spectra barely tell them apart, so the competition
ignores stereochemistry.

### Tautomers

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

**Tautomers** are forms of one molecule that turn into each other by moving a hydrogen (and shifting a
double bond). Chemists treat them as one compound, and the competition normalises tautomers before
computing the key, so they count as the same answer.

## 2. What a mass spectrometer does

A mass spectrometer can only weigh things that carry an electric charge. So it works in two steps.

### Step 1: ionisation

In the ion source, each molecule picks up an **adduct**: a small charged piece that sticks to it. Most
often this is a proton (H+), written **[M+H]+** (M for the molecule). Others in the data are [M+Na]+
(sodium) and [M+NH4]+ (ammonium). The machine then lets through only ions of one mass, the
**precursor m/z**. This mass is measured so precisely that it usually gives the formula.

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

### Step 2: fragmentation

The selected ions are shot into a chamber of gas (the collision cell). When an ion hits a gas molecule
it breaks, and the charge stays on one of the two pieces. Only the **charged** piece can be steered to
the detector and weighed; the **neutral** piece is invisible. Here phenylalanine loses water and carbon
monoxide (46.01 together), and the detector sees 120.08.

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

### The spectrum

One ion breaks once, but the chamber holds millions of them, and a molecule can break in many places.
The detector counts how often each charged mass arrives. That histogram is the **MS/MS spectrum**:
the distribution of charged pieces the molecule breaks into. Each line is a mass-to-charge ratio
(**m/z**; with a charge of +1 this is simply the mass) and its height (**intensity**) says how often
that piece was seen. Here is a real phenylalanine spectrum from the training data:

In [2]:
def intensity_at(r, target):
    mz, it = np.array(r["ms2_mzs"]), np.array(r["ms2_normalized_intensities"])
    hit = np.abs(mz - target) < 0.005
    return it[hit].max() if hit.any() else 0.0


# A clean high-resolution phenylalanine spectrum where both main fragments are visible.
cands = (
    lf.filter((pl.col("inchikey14") == "COLNVLDHVKWLRT") & (pl.col("adduct") == "[M+H]+"))
    .filter(pl.col("instrument_type").str.contains("(?i)orbitrap|qtof|q-tof|qft"))
    .select("ingest_lib", "instrument_type", "precursor_mz", "num_peaks", "ms2_mzs", "ms2_normalized_intensities")
    .sort("num_peaks", "ingest_lib", "precursor_mz")
    .collect()
)
phe = next(r for r in cands.iter_rows(named=True)
           if intensity_at(r, 120.0808) > 0.9 and 0.2 <= intensity_at(r, 103.0542) <= 0.6)
fig, ax, mz, inten = spectrum_axes(
    phe, f"phenylalanine [M+H]+ ({phe['ingest_lib']}, {phe['instrument_type'] or 'instrument unknown'})", xmax=180)
for target, label, xy_text, ha in [(120.0808, "120.08: lost H2O + CO\n(the piece drawn above)", (123, 0.97), "left"),
                               (103.0542, "103.05: then\nalso lost NH3", (107.5, 0.52), "center"),
                               (91.0542, "91.05:\nring + CH2", (88, 0.52), "center"),
                               (77.0386, "77.04:\nthe bare ring", (70.5, 0.52), "center")]:
    i = np.argmin(np.abs(mz - target))
    ax.annotate(label, (mz[i], inten[i]), xytext=xy_text, fontsize=9, va="top", ha=ha,
                arrowprops=None if target > 110 else dict(arrowstyle="-", lw=0.6, color="grey", shrinkA=0, shrinkB=2))
ax.text(phe["precursor_mz"], 0.98, " precursor 166.09\n (intact [M+H]+)", fontsize=9, va="top", color="grey")
show(fig)

## 3. The task

Given the spectra of an unknown molecule, return **up to 25 guesses of its structure**, best first.
The training data has about 2.5 million labelled spectra; the hidden test has about 400 molecules
(roughly 1,500 spectra) measured on one instrument, a Bruker timsTOF ([data page](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data)).

Phenylalanine is small and easy. Here is a more typical molecule from the training data:

In [3]:
fig, ax, _, _ = spectrum_axes(spec, f"{FORMULA}, adduct {spec['adduct']}")
ax.text(spec["precursor_mz"], 0.98, f"precursor {spec['precursor_mz']:.2f} ", fontsize=9, va="top", ha="right", color="grey")
show(fig)

In [4]:
n_same_formula = rows["inchikey14"].n_unique()
display(Markdown(
    f"The precursor m/z gives the formula, here {FORMULA}. But a formula is not a structure: the training "
    f"data alone has **{n_same_formula - 1} other structures** with exactly this formula "
    f"({n_same_formula} in total, counted by InChIKey14)."
))

The precursor m/z gives the formula, here C17H20F2N4O. But a formula is not a structure: the training data alone has **39 other structures** with exactly this formula (40 in total, counted by InChIKey14).

<a id="scoring"></a>

### How it is scored

All spectra of one molecule are scored together, once. Let $r_i$ be the rank of the first guess whose
InChIKey14 matches molecule $i$. The molecule scores the **reciprocal rank**
([evaluation page](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/overview/evaluation), [metric code](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank)):

$$
\text{score}_i = \begin{cases} \dfrac{1}{r_i} & \text{if } r_i \le 25 \\[4pt] 0 & \text{if the answer is not in the top 25} \end{cases}
$$

and the competition score is the mean over all $N$ test molecules, the **mean reciprocal rank at 25**:

$$
\text{MRR@25} = \frac{1}{N} \sum_{i=1}^{N} \text{score}_i
$$

| correct guess at rank | 1 | 2 | 3 | 5 | 10 | 25 | not in top 25 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| points | 1.00 | 0.50 | 0.33 | 0.20 | 0.10 | 0.04 | 0 |

**Example.** Three molecules, answers found at rank 1, at rank 4, and not at all:
MRR@25 = (1 + 0.25 + 0) / 3 = 0.42.

What this means in practice:

- **Rank 1 dominates.** It is worth more than ranks 2 and 3 combined (1 > 0.5 + 0.33). Moving an
  answer from rank 2 to rank 1 gains as much as moving it from "not found" to rank 2.
- **The tail still counts, a little.** Ranks 10 to 25 are worth 0.04 to 0.1 each, so it pays to fill
  all 25 slots with plausible candidates.
- **Only the first hit counts.** Two guesses with the same InChIKey14 (for example two stereoisomers)
  waste a slot ([forum](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/742042)), and invalid SMILES still take up a rank (in the
  [metric code](https://www.kaggle.com/code/metric/casmi-mean-reciprocal-rank); the host confirmed only
  that empty entries are dropped).

### How you submit

Unlike most Kaggle competitions, you never see the test set. You submit a **notebook**, not a file of
predictions. Kaggle runs it on the hidden test data, offline (no internet) and within 9 hours
([code requirements](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/overview/code-requirements)). So
everything your solution needs has to be attached to the notebook in advance: installed packages,
pretrained models, and any extra data such as structure databases. The `test.parquet` you can see is a
placeholder copied from the training data; it is swapped for the real one during the scoring run
([data page](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data)).

The test set is hidden so that nobody can look at the spectra themselves, label them by hand, or tailor
a solution to them.

### Three kinds of test molecule

The host splits the hidden test molecules into three classes and keeps the mix secret
([competition data page](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/data)):

| Class | Definition |
| --- | --- |
| 1: in public spectral libraries | The structure has publicly available reference MS/MS spectra, so it may be found by similarity to the training spectra. |
| 2: known structure, no public spectra | No public spectra exist for this molecule, but the structure is in PubChem or the natural-products database COCONUT, so it is reachable by database retrieval. |
| 3: novel structure | The structure is not in PubChem at all and must be predicted de novo. |

[Leaderboard probing](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/741597) (not official) puts class 1 near 16%: public notebooks
that only do library lookups score about 0.15, and library search scores about 0.93 on class 1
molecules (4.2 finds about 0.92). How the rest splits is a rougher guess, about 45% class 2 and 39%
class 3.

### Strategies

The classes already hint at two ways to approach the challenge:

1. **Retrieve from the training data.** If the molecule has reference spectra (class 1), find the most
   similar training spectra and return their structures.
2. **Add public structure databases.** Bring in structures from PubChem, COCONUT and similar
   collections, and find a way to match a structure to a spectrum that has no reference (class 2).

Class 3 needs **generating novel structures**. From personal experience I found this quite hard, though
not impossible, and no public notebook offers a robust way to do it yet. If the 39% estimate is right,
those molecules are out of reach for retrieval, which caps a retrieval-only pipeline at roughly 0.6,
even if every class 1 and class 2 molecule were ranked first.

So one sound strategy is to build a robust retrieval pipeline. Generation is the other route, but the
generators that do work are most likely ones that have memorised known public molecules and reproduce
them: retrieval in disguise rather than truly novel structures. This split explains most of the design
of the public pipelines.

---

# 4. Building up to the public solution

When I read someone else's solution, I try to ask myself a few questions. How would I have come up with
this? What steps would it have taken me? What was the intuition behind each decision? Pólya's
*How to Solve It* ends with a phase called *looking back*, and one of its questions is "Can you use the
result, or the method, for some other problem?" Asking this is how you learn to spot methods you can
reuse later.

So instead of walking through the top public notebook cell by cell, this section builds it up one idea
at a time: first how to narrow down the candidates by mass, then how to compare two spectra, then how
to score a structure that has no reference spectrum, then how to combine many scores into one ranking,
and finally how to adjust the top of that ranking. At the end, the public notebook should read as these
ideas put together.

## 4.1 Filtering by mass

Before comparing any spectra, cut the candidates down. The cheapest clue is the mass of the molecule,
and the mass spectrometer measures it very precisely.

Masses of molecules are given in **daltons (Da)**. One dalton is one twelfth of the mass of a carbon-12
atom, which is about the mass of one hydrogen atom or one proton. Phenylalanine weighs 165.08 Da. For an
ion with a charge of +1, the m/z value in a spectrum is simply its mass in daltons.

The precursor m/z is the mass of the *ion*, so first subtract the adduct. For [M+H]+ that is one proton,
1.00728 Da:

$$
M = \text{precursor } m/z - 1.00728
$$

Precise masses tell formulas apart because atoms do not weigh whole numbers of daltons. Carbon-12 is
exactly 12 by definition, but hydrogen is 1.00783, nitrogen 14.00307 and oxygen 15.99491. So three
molecules that all weigh "28" are easy to separate:

| molecule | formula | mass (Da) |
| --- | --- | --- |
| carbon monoxide | CO | 27.9949 |
| nitrogen | N2 | 28.0061 |
| ethylene | C2H4 | 28.0313 |

How close is close enough? Instrument accuracy is quoted in **ppm** (parts per million) of the mass:
10 ppm of a 300 Da molecule is 0.003 Da. The top public notebook keeps every structure within ±10 ppm of
the computed mass. Here is what that does for the C17H20F2N4O molecule whose spectrum we looked at
earlier:

In [5]:
ATOM_MASS = {"H": 1.00782503207, "C": 12.0, "N": 14.0030740048, "O": 15.99491461956, "F": 18.99840322,
             "P": 30.97376163, "S": 31.97207100, "Cl": 34.96885268, "Br": 78.9183371, "I": 126.904473,
             "Si": 27.9769265325, "B": 11.0093054, "Se": 79.9165213, "Na": 22.9897692809, "K": 38.96370668}
PROTON = 1.00727646688


def formula_mass(f):
    """Monoisotopic mass from a formula string such as C17H20F2N4O; None if it has unknown elements."""
    if not isinstance(f, str) or not re.fullmatch(r"([A-Z][a-z]?\d*)+", f):
        return None
    total = 0.0
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", f):
        if el not in ATOM_MASS:
            return None
        total += ATOM_MASS[el] * (int(n) if n else 1)
    return total


structures = lf.select("inchikey14", "molecular_formula").unique().collect()
form_mass = {f: formula_mass(f) for f in structures["molecular_formula"].unique().to_list()}
masses = np.array([form_mass[f] if form_mass[f] is not None else np.nan for f in structures["molecular_formula"]])
forms = structures["molecular_formula"].to_numpy()

M = spec["precursor_mz"] - PROTON
true_M = formula_mass(FORMULA)
funnel = [("all structures in the training data", np.isfinite(masses))]
for label, tol in [("within ±0.5 Da (whole-number mass)", 0.5), ("within ±50 ppm", M * 50e-6),
                   ("within ±10 ppm (the public notebook)", M * 10e-6)]:
    funnel.append((label, np.abs(masses - M) <= tol))
funnel.append((f"exactly the formula {FORMULA}", forms == FORMULA))

rows_md = "| filter | structures left | formulas among them |\n| --- | --- | --- |\n"
for label, m in funnel:
    rows_md += f"| {label} | {int(m.sum()):,} | {len(set(forms[m])):,} |\n"
display(Markdown(
    f"The precursor m/z is {spec['precursor_mz']:.4f}, so M = {M:.4f} Da. The true formula {FORMULA} weighs "
    f"{true_M:.4f} Da: the instrument was off by {abs(M - true_M) / true_M * 1e6:.1f} ppm.\n\n" + rows_md))

fig, ax = plt.subplots(figsize=(6.6, 2.6))
labels = [l for l, _ in funnel][::-1]
counts = [int(m.sum()) for _, m in funnel][::-1]
ax.barh(labels, counts, color="#4a6fa5", height=0.6)
ax.set_xscale("log")
for y, c in enumerate(counts):
    ax.text(c * 1.15, y, f"{c:,}", va="center", fontsize=9)
ax.set(xlabel="structures left (log scale)", xlim=(1, max(counts) * 8))
ax.spines[["top", "right"]].set_visible(False)
show(fig)

The precursor m/z is 335.1683, so M = 334.1610 Da. The true formula C17H20F2N4O weighs 334.1605 Da: the instrument was off by 1.5 ppm.

| filter | structures left | formulas among them |
| --- | --- | --- |
| all structures in the training data | 275,309 | 50,762 |
| within ±0.5 Da (whole-number mass) | 2,593 | 231 |
| within ±50 ppm | 516 | 31 |
| within ±10 ppm (the public notebook) | 109 | 5 |
| exactly the formula C17H20F2N4O | 40 | 1 |


Hundreds of thousands of structures become about a hundred, and fixing the formula leaves a few dozen.
Two things limit how far this goes. First, the
filter cannot do better than the formula: **all isomers have the same mass**, so the dozens that are left
have to be told apart by their spectra (next section). Second, the candidates have to be *in* the pool
you search. Here the pool is the training data; the public pipelines add PubChem, COCONUT and other
structure databases, which is how class 2 molecules become reachable at all.

<a id="retrieval"></a>

## 4.2 Retrieval: comparing two spectra

The mass filter leaves a few dozen candidates with the same formula, and now the spectrum has to decide
between them. The easiest case is class 1: the molecule has been measured before, so its reference
spectrum is in the training data. Then the job is a lookup. Compare the unknown spectrum with the
reference spectra of every candidate, and return the structures whose spectra look most alike. All
that is needed is a number for "look alike".

Here is what a good match looks like. The top half is a spectrum of a natural product measured on the
same kind of instrument as the hidden test; the bottom half, mirrored, is the closest reference spectrum
of the same formula from another library. Peaks that line up are coloured.

In [6]:
import math

TOL = 0.01  # Da, as in the top public notebook


def peaks(r):
    """Clean a spectrum the usual way: drop the precursor region and peaks under 1% of the base peak."""
    mz, it = np.asarray(r["ms2_mzs"]), np.asarray(r["ms2_normalized_intensities"], dtype=float)
    if not len(it):
        return {}
    keep = (mz < r["precursor_mz"] - 1.6) & (it >= 0.01 * it.max())
    d = {}
    for m, i in zip(mz[keep], it[keep]):
        k = int(round(m / TOL))
        d[k] = d.get(k, 0.0) + float(i)
    return d


def as_unit(d, power=1.0):
    v = {k: x**power for k, x in d.items()}
    n = math.sqrt(sum(x * x for x in v.values()))
    return {k: x / n for k, x in v.items()} if n else v


def entropy(p):
    return -sum(x * math.log(x) for x in p.values() if x > 0)


def as_distribution(d):
    """Intensities as probabilities; low-entropy spectra get their small peaks boosted (Li et al. 2021)."""
    s = sum(d.values())
    p = {k: x / s for k, x in d.items()} if s else dict(d)
    S = entropy(p)
    if S < 3:
        w = 0.25 + 0.25 * S
        p = {k: x**w for k, x in p.items()}
        s = sum(p.values())
        p = {k: x / s for k, x in p.items()}
    return p, entropy(p)


def cosine(a, b):
    return sum(x * b.get(k, 0.0) for k, x in a.items())


def entropy_similarity(a, b):
    (pa, sa), (pb, sb) = a, b
    mix = {k: x / 2 for k, x in pa.items()}
    for k, x in pb.items():
        mix[k] = mix.get(k, 0.0) + x / 2
    return 1 - (2 * entropy(mix) - sa - sb) / math.log(4)


MEASURES = {
    "cosine": (lambda d: as_unit(d, 1.0), cosine),
    "cosine on square-rooted intensities": (lambda d: as_unit(d, 0.5), cosine),
    "entropy similarity": (as_distribution, entropy_similarity),
}

# Benchmark: every enveda-np-examples molecule against all other libraries, same formula, [M+H]+ only.
cols = ["inchikey14", "molecular_formula", "precursor_mz", "ingest_lib", "instrument_type", "ms2_mzs",
        "ms2_normalized_intensities"]
mh = lf.filter(pl.col("adduct") == "[M+H]+")
np_q = mh.filter(pl.col("ingest_lib") == "enveda-np-examples").select(cols).collect()
np_lib = mh.filter(pl.col("molecular_formula").is_in(np_q["molecular_formula"].unique().to_list())
                   & (pl.col("ingest_lib") != "enveda-np-examples")).select(cols).collect()
lib_by_formula = {}
for r in np_lib.iter_rows(named=True):
    lib_by_formula.setdefault(r["molecular_formula"], []).append((r["inchikey14"], peaks(r), r))
queries = {}
for r in np_q.iter_rows(named=True):
    queries.setdefault((r["inchikey14"], r["molecular_formula"]), []).append((peaks(r), r))

results = {}
for name, (prep, sim) in MEASURES.items():
    rows_out = []
    for (ik, f), qs in sorted(queries.items()):
        qp = [(prep(d), r) for d, r in qs if d]
        best = {}
        for cik, d, cr in lib_by_formula[f]:
            if not d or not qp:
                continue
            cp = prep(d)
            for x, qr in qp:
                s = sim(x, cp)
                if s > best.get(cik, (-1.0,))[0]:
                    best[cik] = (s, qr, cr)
        ranked = sorted(best, key=lambda k: (-best[k][0], k))
        rank = ranked.index(ik) + 1 if ik in ranked else None
        rows_out.append({"ik": ik, "rank": rank, "rr": 1 / rank if rank and rank <= 25 else 0.0,
                         "top_score": best[ranked[0]][0] if ranked else 0.0, "top_ok": rank == 1,
                         "pair": best.get(ik, (None, None, None))[1:], "n_cand": len(best)})
    results[name] = rows_out

# Mirror plot: the first molecule (by key) that every measure gets right, with a few dozen peaks.
ent_rows = {r["ik"]: r for r in results["entropy similarity"]}
cos_rows = {r["ik"]: r for r in results["cosine"]}
demo = next(r for ik, r in sorted(ent_rows.items())
            if r["top_ok"] and cos_rows[ik]["top_ok"] and 15 <= len(peaks(r["pair"][0])) <= 40 and r["n_cand"] >= 5)
qr, cr = demo["pair"]


def stick(r):
    """Cleaned peaks scaled to the tallest one; empty arrays if nothing is left after cleaning."""
    mz, it = np.asarray(r["ms2_mzs"]), np.asarray(r["ms2_normalized_intensities"], dtype=float)
    if not len(it):
        return mz, it
    keep = (mz < r["precursor_mz"] - 1.6) & (it >= 0.01 * it.max())
    if not keep.any():
        return mz[keep], it[keep]
    return mz[keep], it[keep] / it[keep].max()


(qm, qi), (cm, ci) = stick(qr), stick(cr)
q_hit = np.array([np.abs(cm - m).min() <= TOL if len(cm) else False for m in qm])
c_hit = np.array([np.abs(qm - m).min() <= TOL if len(qm) else False for m in cm])
fig, ax = plt.subplots(figsize=(9, 4.2))
for m_, i_, hit, sign in [(qm, qi, q_hit, 1), (cm, ci, c_hit, -1)]:
    ax.vlines(m_[~hit], 0, sign * i_[~hit], lw=1.8, color="#b8bcc4", capstyle="butt")
    ax.vlines(m_[hit], 0, sign * i_[hit], lw=1.8, color="#e07b39", capstyle="butt")
ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="m/z", ylabel="relative intensity", ylim=(-1.08, 1.08),
       xlim=(min(qm.min(), cm.min()) - 10, max(qr["precursor_mz"], cr["precursor_mz"]) + 5),
       title=f"{qr['molecular_formula']}: unknown (top) vs closest reference (bottom)")
ax.set_yticks([-1, -0.5, 0, 0.5, 1], ["1", "0.5", "0", "0.5", "1"])
ax.text(0.01, 0.97, f"unknown: {qr['ingest_lib']} ({qr['instrument_type'] or 'timsTOF'})", transform=ax.transAxes,
        fontsize=9, va="top")
ax.text(0.01, 0.03, f"reference: {cr['ingest_lib']} ({cr['instrument_type'] or 'instrument unknown'})",
        transform=ax.transAxes, fontsize=9, va="bottom")
ax.spines[["top", "right"]].set_visible(False)
show(fig)

### Cosine similarity

Cosine similarity is the classic first tool for retrieval: if you have built a RAG system, you have used
it to compare text embeddings. Here it is applied directly to the spectra, with no learned embedding in
between (that comes later). Each spectrum becomes a list of numbers, one per m/z position (two peaks
count as the same position if they are within 0.01 Da), and cosine measures the angle between the two
lists:

$$
\cos(a, b) = \frac{\sum_i a_i b_i}{\sqrt{\sum_i a_i^2} \, \sqrt{\sum_i b_i^2}}
$$

It is 1 when the two spectra have the same peaks in the same proportions, and 0 when they share no
peaks. Because intensities are multiplied, a few tall peaks dominate the score and small peaks barely
count. A common fix is to take the square root of the intensities first.

### Entropy similarity

[Li et al. (2021)](https://www.nature.com/articles/s41592-021-01331-z) treat a spectrum as a probability
distribution over m/z, with the intensities scaled to sum to 1. Its **spectral entropy**
$S = -\sum_i p_i \ln p_i$ then measures how spread out it is: one dominant peak gives a low entropy, many
peaks of similar height a high one.

To compare two spectra, combine them into one, each at half weight. If they are alike, their peaks land
on top of each other and the combined spectrum is no more spread out than either of them. If they are
different, it has the peaks of both and its entropy goes up. Entropy similarity turns that increase into
a score:

$$
\text{similarity}(a, b) = 1 - \frac{2\,S_{ab} - S_a - S_b}{\ln 4}
$$

where $S_{ab}$ is the entropy of the combined spectrum. It is 1 for identical spectra and 0 for spectra
with no peaks in common. Li et al. also flatten spectra whose intensity sits in just a few peaks, so a
single dominant peak cannot carry the score. In their benchmark this identified compounds better than
cosine similarity (called the "dot product" in mass spectrometry) and dozens of other similarity
measures.

### Does it matter here?

A quick test on the training data: take the 210 natural products in `enveda-np-examples` (the library
closest to the hidden test) and search each one against every other library, among structures of the
same formula. Every one of them has reference spectra elsewhere, so this mimics a class 1 lookup across
instruments. Each measure ranks the candidates; I also check how well the **top score** tells you
whether the top answer is right (AUC: 0.5 is a coin flip, 1.0 is perfect).

In [7]:
def auc(scores, ok):
    pos, neg = scores[ok], scores[~ok]
    if not len(pos) or not len(neg):
        return float("nan")
    return float((pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean())


rng = np.random.default_rng(0)
arr = {n: (np.array([r["rr"] for r in rs]), np.array([r["top_score"] for r in rs]), np.array([r["top_ok"] for r in rs]))
       for n, rs in results.items()}
n_mol = len(arr["cosine"][0])
boot = {"mrr": [], "auc": []}
for _ in range(2000):
    i = rng.integers(0, n_mol, n_mol)
    e, c = arr["entropy similarity"], arr["cosine"]
    boot["mrr"].append(e[0][i].mean() - c[0][i].mean())
    boot["auc"].append(auc(e[1][i], e[2][i]) - auc(c[1][i], c[2][i]))
ci = {k: np.nanpercentile(v, [2.5, 97.5]) for k, v in boot.items()}

cells = {}
for name, (rr, sc, ok) in arr.items():
    hi = sc >= 0.9
    cells[name] = [(rr.mean(), f"{rr.mean():.3f}"), (ok.mean(), f"{ok.mean():.0%}"), (auc(sc, ok), f"{auc(sc, ok):.2f}"),
                   (hi.mean(), f"{hi.mean():.0%}"), (ok[hi].mean(), f"{ok[hi].mean():.0%}")]
n_cols = len(next(iter(cells.values())))
# Bold the best value per column (higher is better). The "top score >= 0.9" share is not a quality
# measure (cosine scores almost everything near 1), so it is never bolded.
NO_BOLD = {3}
best = [None if j in NO_BOLD else max(float(c[j][1].rstrip("%")) for c in cells.values()) for j in range(n_cols)]
table = ("| measure | MRR@25 | right at rank 1 | AUC of the top score | top score ≥ 0.9 | right when ≥ 0.9 |\n"
         "| --- | --- | --- | --- | --- | --- |\n")
for name, row in cells.items():
    shown = [f"**{t}**" if best[j] is not None and float(t.rstrip("%")) == best[j] else t for j, (_, t) in enumerate(row)]
    table += f"| {name} | " + " | ".join(shown) + " |\n"
e_rr, c_rr = arr["entropy similarity"][0].mean(), arr["cosine"][0].mean()
display(Markdown(table + f"""
{n_mol} molecules, median {int(np.median([r['n_cand'] for r in results['cosine']]))} candidate structures each,
peaks matched within {TOL} Da. Entropy minus cosine (95% bootstrap interval): MRR@25
{e_rr - c_rr:+.3f} [{ci['mrr'][0]:+.3f}, {ci['mrr'][1]:+.3f}], AUC
{np.nanmean(boot['auc']):+.2f} [{ci['auc'][0]:+.2f}, {ci['auc'][1]:+.2f}].
"""))

fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, name in zip(axes, ["cosine", "entropy similarity"]):
    _, sc, ok = arr[name]
    bins = np.linspace(0, 1, 21)
    ax.hist([sc[ok], sc[~ok]], bins=bins, stacked=True, color=["#4a6fa5", "#e07b39"],
            label=["top answer right", "top answer wrong"], edgecolor="white", linewidth=0.5)
    ax.set(title=name, xlabel="top score", xlim=(0, 1))
    ax.spines[["top", "right"]].set_visible(False)
    # Arrow to where the wrong answers sit: their median top score, at the top of that bar.
    med = float(np.median(sc[~ok]))
    j = min(np.searchsorted(bins, med, side="right") - 1, len(bins) - 2)
    top = np.histogram(sc, bins=bins)[0][j]
    ax.annotate(f"wrong answers\n(median {med:.2f})", xy=(med, top), xytext=(0.3, 0.55), textcoords="axes fraction",
                fontsize=9, color="#b0561f", ha="center",
                arrowprops=dict(arrowstyle="->", color="#b0561f", lw=1.2, shrinkA=2, shrinkB=2))
axes[0].set_ylabel("molecules")
axes[0].legend(frameon=False, fontsize=9, loc="upper left")
show(fig)

| measure | MRR@25 | right at rank 1 | AUC of the top score | top score ≥ 0.9 | right when ≥ 0.9 |
| --- | --- | --- | --- | --- | --- |
| cosine | **0.921** | **87%** | 0.64 | 76% | 91% |
| cosine on square-rooted intensities | 0.907 | 85% | 0.72 | 40% | 93% |
| entropy similarity | 0.913 | 86% | **0.78** | 24% | **96%** |

210 molecules, median 6 candidate structures each,
peaks matched within 0.01 Da. Entropy minus cosine (95% bootstrap interval): MRR@25
-0.008 [-0.032, +0.014], AUC
+0.14 [+0.03, +0.25].


Two things stand out:

- **Ranking: a tie.** All three measures put the right structure first for
  about 85 to 90% of these molecules, and the differences are within noise. Class 1 is close to solved
  by any sensible similarity.
- **Confidence: entropy wins.** With cosine, the wrong answers score almost as high as the
  right ones (median 0.92); entropy similarity pushes them down (median 0.70), so its score says more
  about whether to trust the match. We will come back to this [later](#library-gate): the top public
  notebook uses
  exactly this score as a gate, to decide when a library match can be trusted and when to bring in
  candidates from other sources.

This only works when a reference spectrum exists. For about 84% of the test molecules (classes 2 and 3)
there is none, which is where the next idea comes in.

## 4.3 From spectrum to structure: fingerprints

Without a reference spectrum, the right structure may well be among the candidates, but there is nothing
to compare the unknown spectrum with. So we need a way to compare a spectrum with a *structure*
directly. There are two directions: translate the spectrum into the language of structures (this
section), or translate each candidate structure into a spectrum (the next one).

### Fingerprints

The language of structures that most pipelines use is the **molecular fingerprint**: a long list of
yes/no bits, each saying whether the molecule contains a certain small piece.

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

Real fingerprints have thousands of bits, and most are not named pieces: a Morgan fingerprint, for
example, gives each atom's neighbourhood (the atom, its neighbours, their neighbours) a bit. It is a
bit like n-grams for text, except that the window grows outward from each atom instead of running
along a sentence. Two fingerprints are compared with the **Tanimoto similarity**, the bits they share
divided by the bits either has:

$$
T(a, b) = \frac{|a \cap b|}{|a \cup b|}
$$

Why a fingerprint? Every candidate has a structure, even one that has never been measured, and a
fingerprint can be computed from any structure in a fraction of a millisecond. So if we can guess the
unknown molecule's fingerprint from its spectrum, we can compare that guess with the fingerprint of
every candidate whose structure we know, measured or not:

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

<div align="center" style="text-align:center"><em>This is what makes class 2 reachable: the structures are in public databases such as PubChem and COCONUT, even though nobody has published their spectra.</em></div>

The top public notebook joins four standard RDKit fingerprints (Morgan with radius 2 and 3, RDKit path
fingerprint, MACCS keys; 10,407 bits) and keeps a selected 6,930 of them.

### Guessing a fingerprint from look-alike spectra

Here is a simple way to get from a spectrum to a fingerprint, with no neural network. Molecules with
similar spectra tend to have similar structures. So find the training spectra that look most like the
unknown one, even if they belong to other molecules, and average the fingerprints of *their*
structures. That average is a guess at the unknown's fingerprint, and the candidates can be ranked by
how well they match it. In machine-learning terms this is **k-nearest-neighbours regression**: the
inputs are spectra, the targets are fingerprint bits, and each bit is predicted as the
similarity-weighted average over the k most similar training spectra.

To test it, take the same 210 natural products as before, but now **remove every spectrum of the true
molecule** from the training data, so it looks like class 2: the structure is a candidate, but it was
never measured. The look-alikes come from a sample of the other training spectra (found with cosine
similarity on square-rooted intensities, for speed), and each guess averages the fingerprints of the 10
best look-alikes, weighted by similarity.

In [8]:
import scipy.sparse as sp
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

NBITS, K_LOOKALIKE, NBINS = 2048, 10, int(2000 / TOL)
fp_cols = ["inchikey14", "normalized_smiles", "molecular_formula", "precursor_mz", "ms2_mzs",
           "ms2_normalized_intensities"]
q43 = mh.filter(pl.col("ingest_lib") == "enveda-np-examples").select(fp_cols).collect().sort("inchikey14", "precursor_mz")
lib43 = (mh.filter((pl.col("ingest_lib") != "enveda-np-examples") & ((pl.col("inchikey14").hash(seed=0) % 5) == 0)
                   & ~pl.col("inchikey14").is_in(q43["inchikey14"].unique().to_list()))
         .select(fp_cols).collect().sort("inchikey14", "precursor_mz"))
in_5pct = (lib43["inchikey14"].hash(seed=0) % 20 == 0).to_numpy()


def spectrum_matrix(df):
    """Rows: spectra as unit vectors over 0.01 Da bins (square-rooted intensities)."""
    r_, c_, v_ = [], [], []
    for i, r in enumerate(df.iter_rows(named=True)):
        mz, it = np.asarray(r["ms2_mzs"]), np.asarray(r["ms2_normalized_intensities"], dtype=float)
        if not len(it):
            continue
        keep = (mz < r["precursor_mz"] - 1.6) & (it >= 0.01 * it.max()) & (mz < 2000)
        b = np.round(mz[keep] / TOL).astype(int)
        r_ += [i] * len(b)
        c_ += b.tolist()
        v_ += np.sqrt(it[keep]).tolist()
    m = sp.csr_matrix((v_, (r_, c_)), shape=(df.height, NBINS))
    m.sum_duplicates()
    norm = np.sqrt(np.asarray(m.multiply(m).sum(axis=1)).ravel())
    norm[norm == 0] = 1
    return sp.diags(1 / norm) @ m


morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NBITS)
_fp_cache = {}


def fingerprint(smiles):
    if smiles not in _fp_cache:
        mol = Chem.MolFromSmiles(smiles)
        bits = np.zeros(NBITS, np.float32)
        if mol is not None:
            DataStructs.ConvertToNumpyArray(morgan.GetFingerprint(mol), bits)
        _fp_cache[smiles] = bits
    return _fp_cache[smiles]


sims43 = (spectrum_matrix(q43) @ spectrum_matrix(lib43).T).tocsr()
lib_smiles = lib43["normalized_smiles"].to_list()
cands43 = (lf.filter(pl.col("molecular_formula").is_in(q43["molecular_formula"].unique().to_list()))
           .group_by("molecular_formula", "inchikey14").agg(pl.first("normalized_smiles")).collect()
           .sort("molecular_formula", "inchikey14"))
cands_by_formula = {}
for r in cands43.iter_rows(named=True):
    cands_by_formula.setdefault(r["molecular_formula"], []).append((r["inchikey14"], r["normalized_smiles"]))
q_rows = {}
for i, r in enumerate(q43.iter_rows(named=True)):
    q_rows.setdefault((r["inchikey14"], r["molecular_formula"]), []).append(i)


def expected_rr(scores, truth):
    """Reciprocal rank (0 beyond 25) of `truth` in a {candidate: score} dict; ties count as a fair coin flip."""
    s = scores[truth]
    above = sum(v > s for v in scores.values())
    tied = sum(v == s for v in scores.values())
    return float(np.mean([1 / r if r <= 25 else 0.0 for r in range(above + 1, above + tied + 1)]))


FP_SCORES = {}  # truth key -> {candidate: Tanimoto to the guess}, for the 20% library (reused in 4.4)


def lookalike_mrr(allowed, store=None):
    """MRR@25 when each guess averages the fingerprints of the best look-alikes among `allowed` library rows."""
    out = []
    for (ik, f), idx in sorted(q_rows.items()):
        guess, wsum = np.zeros(NBITS, np.float32), 0.0
        for i in idx:
            row = sims43.getrow(i)
            ok = allowed[row.indices]
            js, ws = row.indices[ok], row.data[ok]
            for t in np.argsort(-ws)[:K_LOOKALIKE]:
                guess += ws[t] * fingerprint(lib_smiles[js[t]])
                wsum += ws[t]
        guess /= max(wsum, 1e-9)
        score = {c: float(np.minimum(guess, fingerprint(s)).sum() / max(np.maximum(guess, fingerprint(s)).sum(), 1e-9))
                 for c, s in cands_by_formula[f]}
        if store is not None:
            store[ik] = score
        out.append(expected_rr(score, ik))
    return np.array(out)


random_rr = np.array([sum(1 / x for x in range(1, min(25, len(cands_by_formula[f])) + 1)) / len(cands_by_formula[f])
                      for (ik, f) in sorted(q_rows)])
runs = [("random order", random_rr, "")]
for label, allowed in [("look-alikes from 5% of the training molecules", in_5pct),
                       ("look-alikes from 20% of the training molecules", np.ones(lib43.height, bool))]:
    rr = lookalike_mrr(allowed, store=FP_SCORES if allowed.all() else None)
    d = rr - random_rr
    b = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
    runs.append((label, rr, f"{d.mean():+.3f} [{np.percentile(b, 2.5):+.3f}, {np.percentile(b, 97.5):+.3f}]"))

t43 = "| ranking by | MRR@25 | right at rank 1 | gain over random (95% interval) |\n| --- | --- | --- | --- |\n"
for label, rr, gain in runs:
    top1 = f"{np.mean(rr == 1):.0%}" if label != "random order" else f"{np.mean(1 / np.array([len(cands_by_formula[f]) for _, f in sorted(q_rows)])):.0%}"
    t43 += f"| {label} | {rr.mean():.3f} | {top1} | {gain or '(baseline)'} |\n"
display(Markdown(t43 + f"\n{len(random_rr)} molecules, median {int(np.median([len(cands_by_formula[f]) for _, f in sorted(q_rows)]))} "
                 f"candidates each; the true molecule's own spectra are never used. Using every training molecule "
                 f"instead of 20% (too slow to run here) gave 0.698 in my tests."))

| ranking by | MRR@25 | right at rank 1 | gain over random (95% interval) |
| --- | --- | --- | --- |
| random order | 0.434 | 27% | (baseline) |
| look-alikes from 5% of the training molecules | 0.550 | 40% | +0.116 [+0.074, +0.158] |
| look-alikes from 20% of the training molecules | 0.672 | 53% | +0.238 [+0.192, +0.285] |

210 molecules, median 7 candidates each; the true molecule's own spectra are never used. Using every training molecule instead of 20% (too slow to run here) gave 0.698 in my tests.

Here is what that looks like for one of the molecules, a steroid (formula C23H32O6). Its own spectra
are not in the library, so the closest look-alikes are spectra of *other* molecules; what matters is
that they share pieces with the unknown one.

In [9]:
from rdkit.Chem.Draw import rdMolDraw2D


def mol_svg(smiles, w=230, h=170):
    d = rdMolDraw2D.MolDraw2DSVG(w, h)
    d.drawOptions().padding = 0.08
    d.DrawMolecule(Chem.MolFromSmiles(smiles))
    d.FinishDrawing()
    return "data:image/svg+xml;base64," + base64.b64encode(d.GetDrawingText().encode()).decode()


def explain_lookalikes(ik, f, idx, n_show=3):
    """The look-alikes behind one guess, and the top candidates ranked against it."""
    best = {}
    guess, wsum = np.zeros(NBITS, np.float32), 0.0
    for i in idx:
        row = sims43.getrow(i)
        top = np.argsort(-row.data)[:K_LOOKALIKE]
        for j, w in zip(row.indices[top], row.data[top]):
            guess += w * fingerprint(lib_smiles[j])
            wsum += w
            key = lib43["inchikey14"][int(j)]
            if w > best.get(key, (0, None))[0]:
                best[key] = (float(w), lib_smiles[j])
    guess /= max(wsum, 1e-9)
    looks = sorted(best.values(), key=lambda t: -t[0])[:n_show]
    cs = cands_by_formula[f]
    scored = sorted(((np.minimum(guess, fingerprint(s)).sum() / max(np.maximum(guess, fingerprint(s)).sum(), 1e-9), c, s)
                     for c, s in cs), key=lambda t: -t[0])
    rank = [c for _, c, _ in scored].index(ik) + 1
    return looks, scored[:n_show], rank, len(cs)


example = None
for (ik, f), idx in sorted(q_rows.items()):
    looks, top_c, rank, n = explain_lookalikes(ik, f, idx)
    if rank == 1 and 6 <= n <= 30 and len(looks) == 3 and min(w for w, _ in looks) >= 0.5:
        example = (ik, f, looks, top_c, n)
        break

if example:
    ik, f, looks, top_c, n = example
    cell = 'style="text-align:center;padding:4px 10px;vertical-align:top;font-size:13px"'
    head = 'style="text-align:right;padding:4px 10px;vertical-align:middle;font-size:13px;color:#555;width:150px"'
    look_cells = "".join(f'<td {cell}><img src="{mol_svg(s)}" width="200"><br>spectral similarity {w:.2f}</td>'
                         for w, s in looks)
    cand_cells = "".join(
        f'<td {cell}><img src="{mol_svg(s)}" width="200"><br>Tanimoto to the guess {t:.2f}'
        + ('<br><b style="color:#2e7d32">✓ the right answer</b>' if c == ik else "")
        + "</td>" for t, c, s in top_c)
    display(HTML(f"""<div align="center" style="text-align:center"><table style="margin:auto;border-collapse:collapse">
<tr style="background:white"><td {head}><b>closest look-alikes</b><br>(other molecules)</td>{look_cells}</tr>
<tr style="background:white"><td {head}><b>best candidates</b><br>(of {n} with formula {f})</td>{cand_cells}</tr>
</table></div>"""))
    scores = ", ".join(f"{t:.2f}" for t, _, _ in top_c)
    display(Markdown(
        f"The guess puts the right structure first, but the three best candidates score {scores}: close "
        "isomers have very similar fingerprints. The fingerprint finds the right family of structures; "
        "telling its members apart takes more evidence, which is what the next section adds."))

closest look-alikes(other molecules),spectral similarity 0.79,spectral similarity 0.78,spectral similarity 0.76
best candidates(of 12 with formula C23H32O6),Tanimoto to the guess 0.38✓ the right answer,Tanimoto to the guess 0.33,Tanimoto to the guess 0.32


The guess puts the right structure first, but the three best candidates score 0.38, 0.33, 0.32: close isomers have very similar fingerprints. The fingerprint finds the right family of structures; telling its members apart takes more evidence, which is what the next section adds.

Without ever seeing a spectrum of the right molecule, the guessed fingerprint lifts the score well above
random, and more look-alikes give better guesses. The top public notebook uses this exact idea as one of
its signals. Its **analog propagation** (by prvsiyan) finds the 200 training spectra most like the
unknown, using entropy similarity and allowing for a shift in mass, and gives each candidate a feature:
its best Tanimoto similarity to those look-alikes' structures, weighted by how alike the spectra are.

### FPNet: learning the translation

k-nearest neighbours is a crude translator. **FPNet** (FingerPrintNet) learns it instead: a neural
network trained on the training data to predict, from a spectrum, the probability of every fingerprint
bit. This is the learned
embedding promised in the cosine section, with a twist: instead of putting spectra into a space of
their own, it puts them straight into the space of structures (fingerprints), where the candidates already live.

In the top public notebook, FPNet is a transformer that reads the peaks one by one: each peak's m/z, its
distance from the precursor (the **neutral loss**, the mass of the piece that broke off) and its
intensity, plus the precursor mass, collision energy, adduct and instrument. It outputs one score per
fingerprint bit (a logit: how sure it is that the bit is set). The notebook then:

- **averages several models**, each run on every spectrum of the molecule and on one merged spectrum of
  all of them;
- **scores each candidate** by adding up the logits of the bits it has. That sum is the log-likelihood
  of the candidate's fingerprint under the prediction (up to a constant that is the same for every
  candidate), so it asks: how likely is this structure, given the spectrum? A second version divides
  the sum by the square root of the number of bits, so a large molecule cannot win just by having more
  bits;
- **turns the scores into features** for the ranker of the next step: how far a candidate stands above
  the others (z-score), its rank, its gap to the best candidate. Only for PubChem candidates, which skip
  the ranker, is the score used directly (plus a popularity bonus).

## 4.4 From structure to spectrum: simulating fragments

Now the other direction. Instead of guessing the unknown's fingerprint, take each candidate structure and
ask: if this were the molecule, which peaks would its spectrum have? Then check that prediction against
the spectrum we observed. This is where isomers finally come apart. They have the same atoms and the
same mass, but they are wired differently, so they break into different pieces.

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

<div align="center" style="text-align:center"><em>The mirror image of the diagram in 4.3: instead of turning the spectrum into a fingerprint, each candidate is turned into a predicted spectrum and compared with the one we observed.</em></div>

### Breaking bonds by hand

The simplest forward model needs no training at all. Every peak is a charged piece of the molecule
(section 2), so a candidate can only explain the peaks whose masses match one of its own pieces. Break
its bonds in every possible way, one or two at a time, and list the masses of the pieces: those are the
peaks the candidate *could* produce. Hydrogens often move when a bond breaks, so each piece is allowed
to gain or lose up to two of them, and it carries the proton of the [M+H]+ ion. The score is the share
of the observed intensity that these masses explain, with full credit for pieces from a single break
and 0.6 for pieces that need two.

This is the idea behind [MetFrag](https://doi.org/10.1186/1471-2105-11-148), and the top public notebook
has a version of it called
[MetFrag-lite](https://www.kaggle.com/code/ahmedberatozer/casmi26-v7c-e006-fraggap). Here is the steroid
from 4.3 again: the same observed spectrum,
with the peaks each candidate can explain coloured in.

In [10]:
H_ATOM = ATOM_MASS["H"]


def fragment_masses(smiles, max_pair_bonds=40):
    """Neutral masses of the pieces after breaking 1 bond (level 1) or 2 bonds (level 2); intact = level 0."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {}
    n = mol.GetNumAtoms()
    atom_mass = np.array([ATOM_MASS.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H_ATOM for a in mol.GetAtoms()])
    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in mol.GetBonds()]
    adj = [[] for _ in range(n)]
    for i, (a, b) in enumerate(bonds):
        adj[a].append((b, i))
        adj[b].append((a, i))

    def pieces(cut):
        seen, out = [False] * n, []
        for start in range(n):
            if seen[start]:
                continue
            stack, members, seen[start] = [start], [], True
            while stack:
                u = stack.pop()
                members.append(u)
                for v, bi in adj[u]:
                    if bi not in cut and not seen[v]:
                        seen[v] = True
                        stack.append(v)
            out.append(members)
        return out

    levels = {round(float(atom_mass.sum()), 4): 0}
    nb = len(bonds)
    cuts = [(1, {i}) for i in range(nb)]
    if nb <= max_pair_bonds:
        cuts += [(2, {i, j}) for i in range(nb) for j in range(i + 1, nb)]
    for lv, cut in cuts:
        parts = pieces(cut)
        if len(parts) > 1:
            for m in parts:
                levels.setdefault(round(float(atom_mass[m].sum()), 4), lv)
    return levels


def explained(levels, mz):
    """Credit per peak: 1 if a 0- or 1-break piece matches, 0.6 if only a 2-break piece does, else 0."""
    if not levels:
        return np.zeros(len(mz))
    m = np.array(list(levels))
    lv = np.repeat(np.array(list(levels.values())), 5)
    ions = (m[:, None] + PROTON + H_ATOM * np.arange(-2, 3)[None, :]).ravel()
    order = np.argsort(ions)
    ions, lv = ions[order], lv[order]
    lo, hi = np.searchsorted(ions, mz - TOL), np.searchsorted(ions, mz + TOL)
    return np.array([0.0 if h == l else (1.0 if lv[l:h].min() <= 1 else 0.6) for l, h in zip(lo, hi)])


_frag_cache = {}


def fragment_score(smiles, spectra):
    if smiles not in _frag_cache:
        _frag_cache[smiles] = fragment_masses(smiles)
    shares = []
    for r in spectra:
        mz, it = stick(r)
        if len(mz):
            shares.append(float((explained(_frag_cache[smiles], mz) * it).sum() / it.sum()))
    return float(np.mean(shares)) if shares else 0.0


q_spectra = {}
for r in q43.iter_rows(named=True):
    q_spectra.setdefault(r["inchikey14"], []).append(r)
FRAG_SCORES = {ik: {c: fragment_score(s, q_spectra[ik]) for c, s in cands_by_formula[f]} for (ik, f) in sorted(q_rows)}

# Two-panel figure for the steroid example from 4.3: right answer vs its best-scoring rival.
if example:
    ik, f = example[0], example[1]
    smiles_of = dict(cands_by_formula[f])
    rival = max((c for c in FRAG_SCORES[ik] if c != ik), key=lambda c: FRAG_SCORES[ik][c])
    spec_ex = q_spectra[ik][0]
    mz, it = stick(spec_ex)
    fig, axes = plt.subplots(1, 2, figsize=(9, 2.9), sharey=True)
    for ax, c, title in [(axes[0], ik, "the right answer"), (axes[1], rival, "its closest rival")]:
        cr = explained(_frag_cache[smiles_of[c]], mz)
        share = (cr * it).sum() / it.sum()
        ax.vlines(mz[cr == 0], 0, it[cr == 0], lw=1.8, color="#b8bcc4", capstyle="butt")
        ax.vlines(mz[cr > 0], 0, it[cr > 0], lw=1.8, color="#e07b39", capstyle="butt")
        ax.set(title=f"{title}: explains {share:.0%}", xlabel="m/z", ylim=(0, 1.05),
               xlim=(mz.min() - 10, spec_ex["precursor_mz"] + 5))
        ax.spines[["top", "right"]].set_visible(False)
    axes[0].set_ylabel("relative intensity")
    show(fig)
    shares = [(explained(_frag_cache[smiles_of[c]], mz) * it).sum() / it.sum() for c in (ik, rival)]
    display(Markdown(
        f"Both candidates explain about the same share of the spectrum ({shares[0]:.0%} and {shares[1]:.0%}): "
        "close isomers break into mostly the same pieces. The second-tallest peak, at m/z 309.19, is the "
        "molecule losing acetic acid and two waters (96.04 Da) one after another. That takes more than two "
        "breaks, so this simple model cannot explain it for either candidate."))

Both candidates explain about the same share of the spectrum (19% and 30%): close isomers break into mostly the same pieces. The second-tallest peak, at m/z 309.19, is the molecule losing acetic acid and two waters (96.04 Da) one after another. That takes more than two breaks, so this simple model cannot explain it for either candidate.

Over all 210 molecules, in the same class-2-like test as before (the true molecule's spectra are never
used), with the fingerprint guess from 4.3 for comparison:

In [11]:
def mrr_of(scores_by_truth):
    return np.array([expected_rr(scores_by_truth[ik], ik) for (ik, f) in sorted(q_rows)])


def zscores(d):
    v = np.array(list(d.values()))
    sd = v.std() or 1.0
    return {k: (x - v.mean()) / sd for k, x in d.items()}


BOTH = {ik: {c: zscores(FP_SCORES[ik])[c] + zscores(FRAG_SCORES[ik])[c] for c in FP_SCORES[ik]} for ik in FP_SCORES}
rows44 = [("random order", random_rr), ("fingerprint guess (4.3)", mrr_of(FP_SCORES)),
          ("breaking bonds", mrr_of(FRAG_SCORES)), ("both, added up", mrr_of(BOTH))]


def gain(a, b):
    d, r = a - b, np.random.default_rng(0)
    bs = [d[r.integers(0, len(d), len(d))].mean() for _ in range(2000)]
    return f"{d.mean():+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]"


t44 = "| ranking by | MRR@25 | gain over random (95% interval) |\n| --- | --- | --- |\n"
for label, rr in rows44:
    t44 += f"| {label} | {rr.mean():.3f} | {gain(rr, random_rr) if label != 'random order' else '(baseline)'} |\n"
both_vs_fp = gain(rows44[3][1], rows44[1][1])
display(Markdown(t44 + f"\n\"Both, added up\" turns each method's scores into z-scores within the molecule's "
                 f"candidates and adds them. Gain of both over the fingerprint guess alone: {both_vs_fp}."))

| ranking by | MRR@25 | gain over random (95% interval) |
| --- | --- | --- |
| random order | 0.434 | (baseline) |
| fingerprint guess (4.3) | 0.672 | +0.238 [+0.190, +0.286] |
| breaking bonds | 0.600 | +0.166 [+0.123, +0.209] |
| both, added up | 0.664 | +0.230 [+0.182, +0.279] |

"Both, added up" turns each method's scores into z-scores within the molecule's candidates and adds them. Gain of both over the fingerprint guess alone: -0.008 [-0.037, +0.020].

Breaking bonds by hand already beats random clearly, though on its own it does a bit worse than the
fingerprint guess. Added together the two do a little better than either, but the gain is within noise
here. They use different evidence (look-alike molecules versus the candidate's own pieces), so combining
them should help; doing that well is the subject of the next section.

### Learned simulators: ICEBERG and GLACIER

Breaking bonds tells you which masses are *possible*, not which peaks actually appear or how tall
they are: most bonds never break, and a few break almost every time. A **forward model** learns this
from data: give it a structure and it predicts the whole spectrum. We then compare that prediction
with the observed spectrum using entropy similarity, so it is [retrieval](#retrieval) again, against
a simulated reference.

The top public notebook uses two forward models from [ms-pred](https://github.com/coleygroup/ms-pred):
[ICEBERG](https://arxiv.org/abs/2304.13136), which breaks the molecule step by step and then predicts
how tall each piece's peak is, and [GLACIER](https://arxiv.org/abs/2606.29161), a single transformer
that picks the pieces directly and is the stronger of the two. Simulating is slow, so only the top of
each candidate list gets simulated. Bond breaking scores whatever is left.

<a id="n1"></a>

### Other ion types: N1A and the N1 challenger

*I genuinely have no idea what is up with the naming scheme here. N1 and N1A are not clear names; I
would have called them "adduct conversion" and "adduct conversion with both simulators voting".*

*GLACIER only reads spectra where the molecule picked up a proton.* About 8% of the natural products
we tested on were measured only another way (with ammonium attached, or after losing a water), so for
them the stronger simulator stays silent. The fix: edit those spectra so they look like the proton
version of the same molecule, then let GLACIER score them.

A [first version](https://www.kaggle.com/code/seyitkaangunes/casmi26-e006-n1a-cc-by-chebi-lb-0-452) only did this when
ICEBERG had nothing, and it made no difference.
[flexonafft](https://www.kaggle.com/code/flexonafft/casmi26-closing-the-evidence-gap) kept the edited spectra for every such
molecule and added the two simulators' scores together (standardised, like "both, added up" above), so
two models that make different mistakes vote together. It gave the same gain on three notebooks:

| notebook | before | after | change |
| --- | --- | --- | --- |
| [ahmedberatozer](https://www.kaggle.com/code/ahmedberatozer/casmi26-v22-n1-challenger) | 0.455 | 0.463 | <span style="color:#2e7d32"><b>+0.008</b></span> |
| [dmitriigluzdov](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures) | 0.456 | 0.464 | <span style="color:#2e7d32"><b>+0.008</b></span> |
| [flexonafft](https://www.kaggle.com/code/flexonafft/casmi26-closing-the-evidence-gap) | 0.454 | 0.462 | <span style="color:#2e7d32"><b>+0.008</b></span> |
| **average** | | | <span style="color:#2e7d32"><b>+0.008</b></span> |

Each gain alone is within noise, but the same gain three times is the strongest evidence any recent
change has ([what is real](#real-noise)).

## 4.5 Ranking: combining the evidence

By now every candidate has several scores: how well it matches the guessed fingerprint, how much of the
spectrum its pieces explain, and (in the real pipeline) library matches, FPNet scores and simulator
scores. "Both, added up" in 4.4 weighted two of them equally, which is a guess. Some evidence deserves
more weight than others, and how much can depend on the molecule: a fingerprint guess built from close
look-alikes is worth more than one built from distant ones.

### Learning to rank

A **ranker** learns the combination from examples. Every (molecule, candidate) pair becomes one row of
features, labelled 1 if the candidate is the right structure and 0 otherwise. A classifier, here
gradient-boosted decision trees, learns to predict that label, and the candidates of a new molecule are
ranked by the predicted probability. This is the simplest form, called *pointwise* learning to rank.
Other forms compare candidates in pairs or optimise the order of the whole list (LambdaRank is a
well-known example).

For the demo, each row gets the two scores from 4.3 and 4.4 in three forms (the raw score, its z-score
among the molecule's candidates, and its gap to the best candidate), plus three pieces of context: how
many candidates there are, how similar the closest look-alike spectrum is, and how many fingerprint
bits the candidate has. The ranker is trained on 500 other training molecules, set up the same way:
each one's own spectra are the query but never among its look-alikes. It is then tested on the same
210 natural products.

In [12]:
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier

N_TRAIN_MOLS = 500
form_sizes = lf.group_by("molecular_formula").agg(pl.col("inchikey14").n_unique().alias("n")).collect()
ok_forms = form_sizes.filter(pl.col("n").is_between(3, 60))["molecular_formula"].to_list()
train_mols = (lib43.select("inchikey14", "molecular_formula").unique()
              .filter(pl.col("molecular_formula").is_in(ok_forms))
              .with_columns(h=pl.col("inchikey14").hash(seed=2)).sort("h").head(N_TRAIN_MOLS))
train_spec = (lib43.filter(pl.col("inchikey14").is_in(train_mols["inchikey14"].to_list()))
              .group_by("inchikey14", maintain_order=True).head(3))
new_forms = sorted(set(train_mols["molecular_formula"]) - set(cands_by_formula))
for r in (lf.filter(pl.col("molecular_formula").is_in(new_forms))
          .group_by("molecular_formula", "inchikey14").agg(pl.first("normalized_smiles")).collect()
          .sort("molecular_formula", "inchikey14").iter_rows(named=True)):
    cands_by_formula.setdefault(r["molecular_formula"], []).append((r["inchikey14"], r["normalized_smiles"]))

lib_keys = lib43["inchikey14"].to_numpy()
FEATURE_NAMES = ["Tanimoto to the guess", "its z-score", "its gap to the best", "share explained by pieces",
                 "its z-score", "its gap to the best", "log number of candidates", "closest look-alike similarity",
                 "candidate fingerprint bits"]


def rows_for(ik, f, fp_scores, frag_scores, best_sim):
    keys = [c for c, _ in cands_by_formula[f]]
    tan = np.array([fp_scores[c] for c in keys])
    frs = np.array([frag_scores[c] for c in keys])
    nbits = np.array([fingerprint(s).sum() for _, s in cands_by_formula[f]])
    z = lambda v: (v - v.mean()) / (v.std() or 1.0)
    X = np.column_stack([tan, z(tan), tan - tan.max(), frs, z(frs), frs - frs.max(),
                         np.full(len(keys), np.log(len(keys))), np.full(len(keys), best_sim), nbits])
    return keys, X, np.array([c == ik for c in keys])


def features_for(spec_df, sims, exclude_self):
    """Features for every molecule in spec_df; look-alikes exclude the molecule itself when asked."""
    groups = {}
    for i, r in enumerate(spec_df.iter_rows(named=True)):
        groups.setdefault((r["inchikey14"], r["molecular_formula"]), []).append(i)
    spectra = list(spec_df.iter_rows(named=True))
    out = {}
    for (ik, f), idx in sorted(groups.items()):
        guess, wsum, best = np.zeros(NBITS, np.float32), 0.0, 0.0
        for i in idx:
            row = sims.getrow(i)
            keep = lib_keys[row.indices] != ik if exclude_self else np.ones(row.nnz, bool)
            js, ws = row.indices[keep], row.data[keep]
            for t in np.argsort(-ws)[:K_LOOKALIKE]:
                guess += ws[t] * fingerprint(lib_smiles[js[t]])
                wsum += ws[t]
            best = max(best, float(ws.max()) if len(ws) else 0.0)
        guess /= max(wsum, 1e-9)
        fp_s = {c: float(np.minimum(guess, fingerprint(s)).sum() / max(np.maximum(guess, fingerprint(s)).sum(), 1e-9))
                for c, s in cands_by_formula[f]}
        fr_s = {c: fragment_score(s, [spectra[i] for i in idx]) for c, s in cands_by_formula[f]}
        out[ik] = rows_for(ik, f, fp_s, fr_s, best)
    return out


train_feats = features_for(train_spec, (spectrum_matrix(train_spec) @ spectrum_matrix(lib43).T).tocsr(), exclude_self=True)
test_feats = features_for(q43, sims43, exclude_self=False)  # the test molecules are not in the library at all

X_train = np.vstack([v[1] for v in train_feats.values()])
y_train = np.concatenate([v[2] for v in train_feats.values()])
ranker = HistGradientBoostingClassifier(max_depth=4, max_iter=300, learning_rate=0.05, early_stopping=False,
                                        random_state=0).fit(X_train, y_train)


def verdict(a, b, name):
    """Plain words for a paired bootstrap of a - b: clearly better / worse, or about the same."""
    d = a - b
    bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    if lo > 0:
        return f"beats {name} (by {d.mean():.3f})"
    if hi < 0:
        return f"loses to {name} (by {-d.mean():.3f})"
    return f"is within noise of {name}"


def verdicts(a, pairs):
    """Join several verdicts, merging the 'within noise' ones: 'is within noise of X and Y'."""
    v = [verdict(a, b, n) for b, n in pairs]
    noise = [x.removeprefix("is within noise of ") for x in v if x.startswith("is within noise")]
    rest = [x for x in v if not x.startswith("is within noise")]
    if noise:
        rest.append("is within noise of " + " and ".join(noise))
    return " and ".join(rest)


def rr_from(scores, labels):
    return expected_rr(dict(enumerate(scores)), int(np.argmax(labels)))


def rrf(*score_lists, k=3):
    """Reciprocal rank fusion; tied candidates share their average rank."""
    return sum(1 / (k + rankdata(-np.asarray(s), method="average")) for s in score_lists)


res45 = {name: [] for name in ["random order", "fingerprint guess", "breaking bonds", "both, added up",
                               "both, fused by rank (RRF)", "learned ranker"]}
for ik, (keys, X, y) in sorted(test_feats.items()):
    n = len(keys)
    res45["random order"].append(sum(1 / x for x in range(1, min(25, n) + 1)) / n)
    res45["fingerprint guess"].append(rr_from(X[:, 0], y))
    res45["breaking bonds"].append(rr_from(X[:, 3], y))
    res45["both, added up"].append(rr_from(X[:, 1] + X[:, 4], y))
    res45["both, fused by rank (RRF)"].append(rr_from(rrf(X[:, 0], X[:, 3]), y))
    res45["learned ranker"].append(rr_from(ranker.predict_proba(X)[:, 1], y))
res45 = {k: np.array(v) for k, v in res45.items()}

t45 = "| ranking by | MRR@25 | right at rank 1 |\n| --- | --- | --- |\n"
chance_top1 = np.mean([1 / len(keys) for keys, _, _ in test_feats.values()])
best_mrr = max(rr.mean() for rr in res45.values())
for name, rr in res45.items():
    top1 = chance_top1 if name == "random order" else np.mean(rr == 1)
    cells = [name, f"{rr.mean():.3f}", f"{top1:.0%}"]
    if rr.mean() == best_mrr:  # bold the best row
        cells = [f"**{c}**" for c in cells]
    t45 += "| " + " | ".join(cells) + " |\n"
display(Markdown(
    t45 + f"\nTrained on {len(train_feats)} other molecules. The learned ranker "
    f"{verdicts(res45['learned ranker'], [(res45['fingerprint guess'], 'the fingerprint guess alone'), (res45['both, added up'], 'adding the two scores up')])}; RRF "
    f"{verdict(res45['both, fused by rank (RRF)'], res45['both, added up'], 'adding them up')}."))

| ranking by | MRR@25 | right at rank 1 |
| --- | --- | --- |
| random order | 0.434 | 27% |
| **fingerprint guess** | **0.672** | **53%** |
| breaking bonds | 0.600 | 39% |
| both, added up | 0.664 | 51% |
| both, fused by rank (RRF) | 0.671 | 50% |
| learned ranker | 0.650 | 49% |

Trained on 500 other molecules. The learned ranker is within noise of the fingerprint guess alone and adding the two scores up; RRF is within noise of adding them up.

The learned ranker scores highest of all, but its lead over the simpler options is small: with only
two signals there is not much left to learn. A ranker is built for
many features, which is how the top public notebook uses it. The notebook runs two **engines**, each a
complete candidate-scoring system that produces its own ranked list:

- **Engine 1** ([ahmedberatozer's v4n](https://www.kaggle.com/code/ahmedberatozer/casmi26-v4n-inference)) combines library matches, FPNet and other features with LightGBM
  rankers and keeps its top 60 candidates.
- **Engine 2** ([prvsiyan's analog propagation](https://www.kaggle.com/code/prvsiyan/analog-propagation-casmi-2026-baseline)) uses scikit-learn's histogram gradient boosting, as in the demo above, with
  31 features in six families: library matches, look-alike Tanimoto (the analog propagation from 4.3),
  FPNet scores, fragment explanations, how well these agree with each other, and the number of
  candidates. It blends two rankers 88/12, both trained on rows built to mimic class 1 and class 2 test
  molecules.

### Fusing ranked lists

Two engines give two lists, and their scores are not comparable: they come from different models
trained on different rows. What *is* comparable is the rank. **Reciprocal rank fusion (RRF)** gives each
candidate $\frac{1}{k + \text{rank}}$ from each list and adds them up, so a candidate near the top of both
lists wins. For example, with $k = 3$ and equal weights:

| candidate | rank in list 1 | rank in list 2 | fused score |
| --- | --- | --- | --- |
| A | 1 | 6 | 1/4 + 1/9 = 0.361 |
| B | 2 | 2 | 1/5 + 1/5 = **0.400** |
| C | 3 | 3 | 1/6 + 1/6 = 0.333 |

B is nobody's favourite, but both lists rank it high, so it wins. The top public notebook fuses its two
engines as

$$
\text{score} = \frac{1}{3 + \text{rank}_1} + \frac{0.6}{3 + \text{rank}_2}
$$

trusting engine 2 a little less. In the demo above, RRF does worse than adding the z-scores, and that is
expected: our two scores are already comparable after z-scoring, and RRF throws away *how much* better
one candidate is than the next. It is the right tool when the scores cannot be compared, which is
exactly the two-engine case.

Where scores *are* comparable, the top notebook still adds z-scores. When it re-scores the fused list
with the simulators from 4.4, each candidate gets the ranker's score plus the ICEBERG and GLACIER
similarities, each as a z-score among the molecule's candidates, with equal weights.

## 4.6 Priors and gates

Everything so far scores a candidate by how well it fits the spectrum. The last part of the top public
notebook adds two other kinds of knowledge: what we know about a candidate before seeing any spectrum
(a **prior**), and rules for which source of candidates to trust when (**gates**).

### The popularity prior

Among all structures of the right mass, a compound that has been studied a lot is a likelier answer than
one that appears once in a supplier's catalogue. The **popularity prior** turns that into a number: the
logarithm of how many records and papers PubChem has for the structure (the top notebook uses
[this public table](https://www.kaggle.com/datasets/dmitriigluzdov/casmi26-pubchem-popularity-prior)).
It is added to the evidence with a small weight. For candidates from PubChem the score is the FPNet
score (as a z-score) plus the popularity times a weight (0.15 in the top notebook); the engines' own
candidates get a similar nudge, and the 200 most-documented PubChem structures of the formula always
join the PubChem candidates.

There is a catch. Popularity mostly measures whether a compound has been studied, and studied compounds
are the ones with published spectra. In
[aryanawasthiteykik's popularity-trap notebook](https://www.kaggle.com/code/aryanawasthiteykik/casmi26-the-popularity-prior-trap),
popularity alone ranks the right answer first among all PubChem structures within 10 ppm for 67% of
the `enveda-np-examples` molecules, which all have public spectra, but for almost none of the
structures without public spectra. It still helps on the leaderboard (+0.075 in that notebook, from
0.146 to 0.221), but far less than such a validation suggests, and once it is in, re-tuning its weight
moves the score by less than the noise (5.2).

<a id="wrong-spectrum"></a>

### How much does the spectrum add?

Popularity scores a candidate without looking at the spectrum, and so does the mass filter. So how much
of a ranking comes from the spectrum itself? One way to find out: hand each molecule the *wrong*
spectrum. Keep its candidates (the same-formula structures left after filtering by mass, 4.1), but
score them against the spectra of a different molecule of nearly the same mass. Whatever still lands
at rank 1 owes nothing to the molecule's own spectrum.

Here that is for the fingerprint guess and breaking bonds added up (4.4), on the same 210 molecules and
in the same class-2-like test. This demo has no popularity data, so its floor is the mass filter alone;
in the public pipeline, popularity lifts that floor.

In [13]:
def lookalike_guess(idx):
    """Fingerprint guess from the look-alikes of the spectra in rows `idx` of q43 (all of the 20% library)."""
    guess, wsum = np.zeros(NBITS, np.float32), 0.0
    for i in idx:
        row = sims43.getrow(i)
        for t in np.argsort(-row.data)[:K_LOOKALIKE]:
            guess += row.data[t] * fingerprint(lib_smiles[row.indices[t]])
            wsum += row.data[t]
    return guess / max(wsum, 1e-9)


def tanimoto_to(guess, smiles):
    fp = fingerprint(smiles)
    return float(np.minimum(guess, fp).sum() / max(np.maximum(guess, fp).sum(), 1e-9))


def p_rank1(scores, truth):
    """Chance that `truth` is at rank 1; ties count as a fair coin flip."""
    s = scores[truth]
    if any(v > s for v in scores.values()):
        return 0.0
    return 1 / sum(v == s for v in scores.values())


keys = sorted(q_rows)
prec = {ik: float(np.median([r["precursor_mz"] for r in q_spectra[ik]])) for ik, _ in keys}
# Donor: the molecule with the nearest precursor m/z among those with a different formula.
donor = {ik: min((k for k in keys if k[1] != f), key=lambda k: (abs(prec[k[0]] - prec[ik]), k[0])) for ik, f in keys}
WRONG = {}
for ik, f in keys:
    d_ik, d_f = donor[ik]
    guess = lookalike_guess(q_rows[(d_ik, d_f)])
    fp_s = {c: tanimoto_to(guess, s) for c, s in cands_by_formula[f]}
    fr_s = {c: fragment_score(s, q_spectra[d_ik]) for c, s in cands_by_formula[f]}
    WRONG[ik] = {c: zscores(fp_s)[c] + zscores(fr_s)[c] for c in fp_s}

gap = np.median([abs(prec[donor[ik][0]] - prec[ik]) for ik, _ in keys])
rows_ws = [("no spectrum (random order)", random_rr, np.mean([1 / len(cands_by_formula[f]) for _, f in keys])),
           ("another molecule's spectrum", mrr_of(WRONG), np.mean([p_rank1(WRONG[ik], ik) for ik, _ in keys])),
           ("its own spectrum", mrr_of(BOTH), np.mean([p_rank1(BOTH[ik], ik) for ik, _ in keys]))]
t_ws = "| candidates scored against | MRR@25 | right at rank 1 |\n| --- | --- | --- |\n"
for label, rr, top1 in rows_ws:
    t_ws += f"| {label} | {rr.mean():.3f} | {top1:.0%} |\n"
display(Markdown(t_ws + f"\nThe borrowed spectra come from a molecule with a different formula whose precursor is a median "
                 f"{gap:.2f} Da away. Own spectrum over the wrong one: {gain(rows_ws[2][1], rows_ws[1][1])} MRR@25 "
                 f"(95% interval); wrong spectrum over random order: {gain(rows_ws[1][1], random_rr)}."))

| candidates scored against | MRR@25 | right at rank 1 |
| --- | --- | --- |
| no spectrum (random order) | 0.434 | 27% |
| another molecule's spectrum | 0.418 | 24% |
| its own spectrum | 0.664 | 51% |

The borrowed spectra come from a molecule with a different formula whose precursor is a median 0.96 Da away. Own spectrum over the wrong one: +0.246 [+0.193, +0.302] MRR@25 (95% interval); wrong spectrum over random order: -0.016 [-0.048, +0.016].

A borrowed spectrum does no better than random order: the two scorers carry no preference of their own
for some kinds of structure. Everything above random comes from the molecule's own spectrum, and it
roughly doubles how often the right answer is first.

<a id="library-gate"></a>

### Gates: when to trust which source

The pipeline draws candidates from two pools. The first is scored by the two engines: the training
data, COCONUT, ChEBI and LIPID MAPS (curated databases of biological molecules and of lipids) and
PubChemLite (a curated subset of about 386,000 PubChem structures). The second is the whole of PubChem,
over a hundred million structures, scored by FPNet in a separate **PubChem channel**. PubChem has far
more of the right answers for class 2, and far more wrong ones too. The gates decide how much room it
gets:

- **Trust a good library match.** If the best library match scores at least 0.9 on entropy similarity,
  engine 1's list stands and no PubChem candidates are mixed in. In the 4.2 test, library matches that
  scored at least 0.9 were right 96% of the time.
- **Otherwise, give PubChem fixed slots.** Below that threshold, PubChem candidates are placed into fixed
  positions in the list rather than mixed in by score, since the two pools' scores are not comparable
  (the same reason RRF exists, 4.5).
- **Promote to rank 1 only on strong evidence.** A PubChem candidate takes the top slot only if FPNet
  prefers it by a clear margin, or if it is both well supported and very popular. Rank 1 is worth more
  than ranks 2 and 3 together ([How it is scored](#scoring)), so it is guarded the most.
- **Hedge at rank 2.** Finally, the top candidate of engine 2 is moved to rank 2, unless it is already at
  rank 1. If the two engines disagree, engine 2's favourite still gets rank 2, worth half a point.

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

<div align="center" style="text-align:center"><em>The three cases of the gate. PubChem gets every other slot when its best candidate beats engine 1's best on FPNet by a wide margin, and every fourth slot otherwise.</em></div>

## 4.7 End-to-end deep learning: one network that proposes and ranks many answers

Everything in 4.1 to 4.6 is a **pipeline**: a pool of candidates from databases, several separate scorers,
a ranker, a fusion rule, gates. Each stage is hand-built and the pool caps what can be found. The
alternative is an **end-to-end** system: one trained network that reads the spectrum and *writes* structures,
and whose output is a ranked list rather than a single answer. That is the only way to reach class 3
molecules (section 3), which are in no database.

Why a list? The scoring rewards it ([How it is scored](#scoring)): up to 25 guesses, and the first hit counts.
A generator is naturally probabilistic, so asking it for many samples and ordering them is the same
"predict more than one solution" idea as MRR@25.

### The common recipe

Almost all of these systems share three steps:

1. **Encode the spectrum** (peaks, neutral losses, precursor mass; a transformer, as in FPNet) into an
   embedding, often a predicted fingerprint.
2. **Decode a structure** from that embedding, constrained to the known formula, and do it many times:
   beam search (keep the k best partial answers) or repeated random sampling.
3. **Clean and rank**: drop invalid SMILES, merge duplicates (by InChIKey14, because only the first hit
   counts), then order the survivors by a score.

### Examples

| system | decoder | how the list is ranked |
| --- | --- | --- |
| [MSNovelist](https://pmc.ncbi.nlm.nih.gov/articles/PMC9262714) | LSTM writing SMILES from a predicted fingerprint and formula; beam search, width about 128 | re-ranked by how well each candidate's own fingerprint matches the predicted one |
| DiffMS | graph diffusion: starts from noise and denoises atoms and bonds, conditioned on the spectrum embedding | several samples per spectrum, ordered afterwards |
| MADGEN, Spec2Mol, MS2Mol | transformer or graph decoders conditioned on the spectrum | beam or sampling, then a score |

Notice the last step of the first row: MSNovelist ranks by **fingerprint match**, exactly the scorer
from 4.3. The generator proposes, the 4.3 scorer judges. In this notebook's terms, an end-to-end system is
often "candidate pool = whatever the decoder wrote" plus the ranking idea you already know.

Reported results (from the MSNovelist paper, on 3,863 GNPS spectra with the true structure held out
of training): the correct structure ranked first for 25% and appeared in the top 128 for 45%, against 39%
at rank 1 for database search. So generation recovers a good share of answers, but the top of the list is
still less reliable than lookup when the answer *is* in a database.

### Why the difference between rank 1 and rank 25 matters here

With MRR@25 a generator whose right answer sits at rank 6 earns 0.17 even though it "found" it. Two
practical consequences:

- **Diversity beats near-duplicates.** Ten samples that differ only in a methyl position are ten chances
  to be wrong in the same way. Deduplicate and, if needed, penalise candidates too similar to ones already
  listed.
- **Ranking is as important as generating.** Good systems oversample, then rerank with a second model
  (fingerprint match, a spectrum simulator such as ICEBERG or GLACIER from 4.4, or both). That is the 4.5
  ranker applied to generated candidates.

### Why it is harder than it looks

A recent audit, [MassSpecGym in the Wild](https://arxiv.org/pdf/2606.19624), found evaluation problems in
at least 17 of 26 papers using that benchmark. Two matter here:

- **Memorisation.** Decoders pretrained on large molecule collections excluded test molecules only by exact
  InChIKey, and did better when close analogues (Tanimoto at least 0.7) were in pretraining. So some
  "novel" generation is retrieval from the decoder's memory, which is the worry raised in section 3.
- **Bugs inflate numbers.** One padding-mask bug turned a 10.73% top-1 into 31.75% for the same model
  family, and the buggy checkpoint was reused by later papers.

Treat published top-1 numbers as an upper bound, and judge a generator by what it does on molecules with no
close neighbour in the training data, which is what class 3 is.

### What this means for this competition

I know of no public notebook that makes a generator pay off on the leaderboard yet (see 3). A realistic
use today is small: let a generator add a few candidates of the right formula to the pool, then let the
existing rankers and gates decide whether to trust them. Because the pool caps class 2 and 3 scores
(section 5), even a handful of correct extra candidates is where such a system would add, if anywhere.


# 5. Putting it together: what gives it its edge

Here is the top public notebook with every idea from section 4 in place. It is
[ahmedberatozer v22](https://www.kaggle.com/code/ahmedberatozer/casmi26-v22-n1-challenger) (0.463), which is
[ahmedberatozer v15](https://www.kaggle.com/code/ahmedberatozer/casmi26-v15-n1a) with the N1 challenger added.
[dmitriigluzdov](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures)
scores 0.001 higher (0.464) with the same change on a separate line of notebooks; that gap is far inside noise.
The notebooks at the top today (0.477) still run this pipeline unchanged and add one stage after it, a
second opinion on the final 25 ([The last stage](#last-stage)).

<div align="center" style="display:flex;justify-content:center;margin:1em 0"></div>

<div align="center" style="text-align:center"><em>The whole pipeline of the top public notebook. Orange: the three scorers (two engines on the pool, the PubChem channel on all of PubChem); green: combining them. The tag on each box is the section that explains it. Simplified: the simulators also re-score the PubChem list before it reaches the gate.</em></div>

The notebook itself is organised in nine numbered stages. If you open it, this is where each one is
explained here:

| stage in the notebook | what it does | explained in |
| --- | --- | --- |
| 1. Setup | installs packages offline, mounts about 15 datasets | 3 (How you submit) |
| 2. Commit switch | a quick test run when you save, the full run only during scoring | not covered above |
| 3. Engine 2: two-ranker engine | look-alikes, FPNet, fragments, two gradient-boosted rankers | 4.2 to 4.5 |
| 4. PubChem-only channel with a popularity prior | FPNet scores for PubChem structures, plus popularity | 4.3, 4.6 |
| 5. Engine 1 | library match, FPNet, LightGBM rankers | 4.2, 4.3, 4.5 |
| 6. Main candidate lists + pool popularity re-rank | the popularity nudge for the engines' candidates | 4.6 |
| 7. ICEBERG + GLACIER spectrum simulators | predicted spectra, entropy similarity, N1A, N1 challenger | 4.4 |
| 8. Gated PubChem merge | the library gate and the PubChem slots | 4.6 |
| 9. Two-engine fusion + final simulator re-rank | RRF, GLACIER re-scoring, MetFrag-lite, rank-2 hedge | 4.5, 4.4, 4.6 |

## 5.1 The lineage

Most of the top public notebooks are this one pipeline, forked many times with small changes. (The
leaderboard itself goes higher, to about 0.52; those teams have not published their code.)
Every score here is a verified score (the notebook's own linked submission on the public leaderboard),
as of 2026-10-29.

| public score | notebook | change from the one above |
| --- | --- | --- |
| 0.451 | [ahmedberatozer v13](https://www.kaggle.com/code/ahmedberatozer/casmi26-v13-chebi-lmsd) / [nursrijan Apex Dossier](https://www.kaggle.com/code/nursrijan/casmi26-the-apex-dossier-tuned-promo-rel150-e2r2) | the shared base |
| 0.455 | [ahmedberatozer v14](https://www.kaggle.com/code/ahmedberatozer/casmi26-v14-pubchemlite) | + [PubChemLite](https://www.kaggle.com/datasets/franciscoangulo/casmi26-pubchemlite-cc0-candidate-fingerprints) in engine 2's pool |
| 0.455 | [ahmedberatozer v15](https://www.kaggle.com/code/ahmedberatozer/casmi26-v15-n1a) | + N1A (first published by [seyitkaangunes](https://www.kaggle.com/code/seyitkaangunes/casmi26-e006-n1a-cc-by-chebi-lb-0-452)) |
| 0.456 | [guhongbin v465](https://www.kaggle.com/code/guhongbin/casmi26-v465-v15-pop015) | popularity weight 0.25 to 0.15 (a side branch) |
| 0.463 | [ahmedberatozer v22](https://www.kaggle.com/code/ahmedberatozer/casmi26-v22-n1-challenger) | v15 + the N1 challenger (from [flexonafft](https://www.kaggle.com/code/flexonafft/casmi26-closing-the-evidence-gap), 0.454 to 0.462) |
| 0.464 | [dmitriigluzdov v16](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures) | the N1 challenger on dmitriigluzdov's own line (0.456 to 0.464) |
| 0.465 | [dmitriigluzdov](https://www.kaggle.com/code/dmitriigluzdov/casmi-26-the-data-behind-spectra-to-structures), later version | a fix to how a PubChem candidate is promoted to rank 1 ([Gates](#library-gate)); this frozen version is the base of everything below |
| 0.466 | [sanpier](https://www.kaggle.com/code/sanpier/pubchem-neural025-lb-0-466) | + a neural network that scores spectrum against structure, fused into the final 25 at weight 0.25 |
| 0.468 | [kozykappa](https://www.kaggle.com/code/kozykappa/lb-0-468-balanced-acquisition-evidence) | rank 1 as before; ranks 2 to 25 at neural weight 0.50 (the recipe of [romanrozen](https://www.kaggle.com/code/romanrozen/0-468-casmi-from-peaks-to-structures)) |
| 0.472 | [cngnguynhuuz fork](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-lb472) of [alexchilton](https://www.kaggle.com/code/alexchilton/casmi26-lb-0-477-0-472-weights-fragment-check) | ranks 2 to 25 at neural weight 0.75 |
| 0.477 | [busyaprime](https://www.kaggle.com/code/busyaprime/lb-0-477-two-0-472-fixes-merged-the-right-way) | + a fragment check on rank 1 (from [romantamrazov's Spectral Parallax](https://www.kaggle.com/code/romantamrazov/casmi-2026-spectral-parallax)) |

<a id="last-stage"></a>

### The last stage: a second opinion at low weight

The last four rows change nothing inside the pipeline. They take its final 25 and re-order them.

The **neural scorer** is a network trained from scratch on the training spectra (no outside data) to say
how well a structure fits a spectrum. It gives its own ranking of the 25, and RRF (4.5) fuses the two
rankings: a candidate at pipeline rank $b$ and neural rank $n$ scores

$$
\frac{1}{3 + b} + \frac{w}{3 + n}
$$

Rank 1 is chosen with $w = 0.25$. At that weight the network can only take the lead from the pipeline's
favourite when it ranks that favourite very low (18th or worse). Ranks 2 to 25 are then re-sorted with a
larger weight, 0.75 in the current top notebooks, so there the network gets a real say.

The **fragment check** is breaking bonds by hand (4.4), untrained. When the library match is weak, a
same-formula isomer from ranks 2 to 6 may take rank 1 if it explains clearly more of the spectrum's
intensity than the current leader, the neural scorer also prefers it, and the engines or the simulators
back it. The old leader moves to rank 2.

Both are cautious about rank 1 for the reason in [How it is scored](#scoring): it is worth more than ranks
2 and 3 together.

<a id="real-noise"></a>
## 5.2 What is real and what is noise

The hidden test has about 400 molecules, and the public leaderboard scores only part of them (about a
third, by community estimates, so about 130 molecules). One molecule moving to rank 1 changes the
public score by up to 1/130, about 0.008. For two fairly different submissions, one standard error of
the *difference* is about 0.016 ([forum estimate](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra/discussion/743254)); for near-identical forks it
is smaller, but then a single molecule is already most of the gap. Every step in the lineage is a few
thousandths. So which of them are real?

- **Probably real: a bigger candidate pool.** Adding PubChemLite helped on three different bases: +0.004
  on v13 (0.451 to 0.455), +0.008 on seyitkaangunes's 0.446 base (to 0.454 in
  [guhongbin v459](https://www.kaggle.com/code/guhongbin/casmi26-v459-e006-defaultimg)), and +0.004 on
  su2567's own pipeline (0.451 to 0.455 in
  [su2567 v22](https://www.kaggle.com/code/su2567/casmi26-v22-pubchemlite)). Each step alone is within
  noise. Three gains in a row is encouraging, not proof: chance alone makes all three positive one time
  in eight, and more often than that, since the three bases share much of their code.
- **Probably real: the N1 challenger.** +0.008 on three bases (details in
  [Other adducts](#n1)). Stronger evidence than PubChemLite's, with the same caveats: each step
  alone is within noise, and three positives happen by chance about one time in eight, more often
  given shared code.
- **Real: the rank-1 weight of the neural scorer.** One author ran the 0.472 notebook with only the
  rank-1 weight changed: 0.25 gives [0.472](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-lb472),
  0.5 gives [0.448](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-wt05) and 1.0 gives
  [0.359](https://www.kaggle.com/code/cngnguynhuuz/casmi26-fork-wt10). A drop of 0.11 means at least
  15 molecules losing rank 1, far beyond noise. A second opinion helps only while it rarely overrules the
  first.
- **Probably real, but small: the last stage as a whole.** 0.465 to 0.477 is +0.012 in four steps, each
  adding a different piece; no single step is beyond noise. About fifteen
  notebooks now show 0.477, but they run the same code on the same test, so they are copies, not
  replications.
- **Noise: most other recent changes.** Five-fold FPNet (0.464 to 0.464), a sixth PubChem slot at rank 24
  (0.464 to 0.464), an extra rule for promoting PubChem to rank 1 (0.464 to 0.464), and forcing rank-1
  answers to differ across molecules (0.463 to 0.463).
- **Noise: the popularity weight.** guhongbin tried four values on the same notebook:

In [14]:
sweep = [(0.05, 0.448), (0.15, 0.456), (0.25, 0.455), (0.35, 0.447)]
fig, ax = plt.subplots(figsize=(6.4, 2.8))
best = max(sc for _, sc in sweep)
low = min(sc for _, sc in sweep)
ax.annotate("", xy=(0.4, low), xytext=(0.4, best), arrowprops=dict(arrowstyle="|-|", color="#4a6fa5", lw=1.2))
ax.text(0.41, best + 0.0012, f"spread {best - low:.3f}: about one molecule\nmoving to rank 1 out of ~130 (0.008)",
        fontsize=8.5, color="#4a6fa5", va="bottom", ha="right")
ax.plot([w for w, _ in sweep], [sc for _, sc in sweep], "o-", color="#e07b39", lw=1.4, ms=6)
for w, sc in sweep:
    ax.annotate(f"{sc:.3f}", (w, sc), xytext=(0, 7), textcoords="offset points", ha="center", fontsize=9)
ax.set(xlabel="popularity weight", ylabel="public score", xticks=[w for w, _ in sweep],
       ylim=(low - 0.004, best + 0.0085), xlim=(0.0, 0.43))
ax.spines[["top", "right"]].set_visible(False)
show(fig)

All four scores lie within 0.009 of each other, about one molecule moving to rank 1. A sweep like this
cannot tell the four values apart, so the top spot is a one-line change that is most likely luck.

Over library-only notebooks (about 0.15, section 3), the top notebook's edge is the class 2 machinery of
section 4: fingerprints, simulators, rankers and gates. Among the forks at the very top, it is mostly
**coverage**: more of the right answers are in the candidate pool to begin with. Everything in section 4
can only rank the structures that made it into the pool, and a class 2 molecule whose structure is
missing scores zero, however good the ranking. The N1 challenger added a second kind of coverage: simulator
evidence for molecules measured only with other adducts. The last stage since then re-orders what is
already in the list.

<p align="center" style="text-align:center"><em>I will keep this notebook up to date as the public leaderboard
moves; the changelog below says what changed.</em></p>

<p align="center" style="text-align:center">If you found this helpful so far, an upvote helps others find it
and keeps me writing.</p>

# Changelog

- **2026-11-11**: 4.7 end-to-end deep learning systems that generate and rank many candidates (the common recipe, MSNovelist, why ranking and diversity matter, evaluation pitfalls).
- **2026-10-29**: leaderboard update to 0.477: the PubChem promotion fix, the neural scorer and the
  fragment check as a last stage on the final 25, with the rank-1 weight sweep; a wrong-spectrum test of
  how much the spectrum adds.
- **2026-10-22 (3)**: simulators and the N1 challenger rewritten at half the length: problem, idea,
  did it work; budgets, adduct mechanics and footnotes moved out.
- **2026-10-22 (2)**: N1A and the N1 challenger get their own subsection (problem, idea, two
  attempts, results), with numbered references.
- **2026-10-22**: leaderboard update to 0.463/0.464: the N1 challenger (keep GLACIER proxy
  scores, re-rank with ICEBERG and GLACIER together), +0.008 on three notebooks; new top notebook,
  lineage rows, noise list.
- **2026-10-20 (11)**: fact-check against the forum: the popularity-trap source and its leaderboard
  gain (+0.075), how the class mix was derived, noise for near-identical forks, N1A credited to
  seyitkaangunes, invalid SMILES sourced to the metric code. Title matches the Kaggle name.
- **2026-10-20 (10)**: review pass on 4.5 to 5: corrected claims about the popularity sweep and the
  PubChemLite evidence, a popularity-sweep plot, the library gate added to the slots figure, an RRF
  worked example, plainer wording, code names replaced by numbers.
- **2026-10-20 (9)**: section 5: the whole pipeline in one diagram, the notebook's nine stages mapped to
  this notebook, the family tree, and what is real versus noise (coverage is the edge).
- **2026-10-20 (8)**: 4.6 priors and gates: the popularity prior and its trap, the library gate,
  PubChem slots, rank-1 promotion and the rank-2 hedge.
- **2026-10-20 (7)**: 4.5 ranking: a learned ranker trained live on other molecules, reciprocal rank
  fusion, the two engines of the top notebook. Takeaway paragraphs removed.
- **2026-10-20 (6)**: 4.4 from structure to spectrum: breaking bonds by hand (MetFrag-lite) on the
  class-2-like test, combined with the fingerprint guess; ICEBERG, GLACIER and N1A explained. Ties now
  count as a fair coin flip in all rankings.
- **2026-10-20 (5)**: 4.3 from spectrum to structure: fingerprints, a look-alike fingerprint
  demo on a class-2-like test (k-nearest neighbours, with a worked example), FPNet.
- **2026-10-20 (4)**: section 4 starts: building up to the public solution; 4.1 filtering by mass
  (daltons, ppm, the candidate funnel); 4.2 retrieval (cosine vs entropy similarity, benchmark on 210
  natural products).
- **2026-10-20 (3)**: sections on the public pipeline, family tree and noise taken out while I rewrite
  them; source links added throughout.
- **2026-10-20 (2)**: "The task" now covers how you submit (hidden test, offline notebook), the host's
  class definitions, and what the classes suggest about strategy.
- **2026-10-20**: key comparisons show a verdict bracket (green ✓ same answer, red ✗ different
  answers); differing letters in red, ignored key blocks muted.
- **2026-10-19 (6)**: scoring moved under "The task", with the formula and a worked example; plots
  centred, y axis starts at 0.
- **2026-10-19 (5)**: mass spectrometer section split into two steps, each with its own figure.
- **2026-10-19 (4)**: new section "What a mass spectrometer does" (ionisation, fragmentation, why only
  charged pieces are seen, a real phenylalanine spectrum); "Words you need" now comes first.
- **2026-10-19 (3)**: sharp SVG figures, centred; each molecule shown as drawing, InChIKey14 (differing
  letters highlighted), description; formulas explained.
- **2026-10-19 (2)**: prose first, no code shown. New section "Words you need" with figures for
  isomers, stereochemistry, tautomers and InChIKey14; scoring table; pipeline diagram.
- **2026-10-19**: first draft. Task, metric, classes, toy library match on one molecule, pipeline
  stages and glossary, family tree up to 0.456.